# 04_cfar_calculation.ipynb
## Расчёт Cash Flow at Risk тремя методами

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

print("=== Cash Flow at Risk Calculation ===\n")

In [ ]:
# Загружаем дневные потоки
daily = pd.read_csv("daily_cashflow.csv")
net_flows = daily['net_flow'].values

print(f"📈 Загружено {len(net_flows)} дней")
print(f"   Средний дневной поток: {np.mean(net_flows):,.2f}")
print(f"   Стандартное отклонение: {np.std(net_flows):,.2f}")
print(f"   Минимум: {np.min(net_flows):,.2f}")
print(f"   Максимум: {np.max(net_flows):,.2f}\n")

In [ ]:
# 1. Исторический VaR (95%)
conf_level = 0.95
historical_var = np.percentile(net_flows, (1 - conf_level) * 100)

# 2. Параметрический VaR
mu = np.mean(net_flows)
sigma = np.std(net_flows)
parametric_var = mu + sigma * norm.ppf(1 - conf_level)

# 3. Монте-Карло (бутстрап)
n_simulations = 10000
simulated_flows = np.random.choice(net_flows, size=(n_simulations, len(net_flows)), replace=True)
simulated_annual = simulated_flows.sum(axis=1)
monte_carlo_var = np.percentile(simulated_annual, (1 - conf_level) * 100)

print(f"📊 Результаты (95% доверительный уровень):")
print(f"   Исторический VaR: {historical_var:,.2f}")
print(f"   Параметрический VaR: {parametric_var:,.2f}")
print(f"   Монте-Карло VaR: {monte_carlo_var:,.2f}")

In [ ]:
# Сохраняем результаты
results = f"""Cash Flow at Risk (95% confidence)

Исторический VaR: {historical_var:,.2f}
Параметрический VaR: {parametric_var:,.2f}
Монте-Карло VaR: {monte_carlo_var:,.2f}
"""

with open("results.txt", "w") as f:
    f.write(results)

print("\n✅ Результаты сохранены в results.txt")

In [ ]:
# Строим график
plt.figure(figsize=(10, 6))
plt.hist(net_flows, bins=50, alpha=0.7, color='blue', edgecolor='black', label='Дневные потоки')
plt.axvline(historical_var, color='red', linestyle='--', linewidth=2, label=f'VaR 95%: {historical_var:,.0f}')
plt.xlabel('Дневной денежный поток')
plt.ylabel('Частота')
plt.title('Распределение дневных денежных потоков с границей CFaR 95%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('cfar_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ График сохранён в cfar_distribution.png")